<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-7_Fixed_RAG_vs_Workflow_RAG_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 7 — Fixed RAG vs Workflow RAG

**학습 목표**
1. `retrieve → generate`의 **Fixed RAG**를 구현합니다.
2. `rewrite → retrieve → judge → (최대 3회 반복) → generate`의 **Workflow RAG**를 구현합니다.
3. `judge()`를 LLM으로 바꾸어도 개발자가 루프와 경로를 고정했다면 **Workflow**라는 점을 설명합니다.

**실행 환경:** Google Colab / Python 3. 기본 모드는 API 키 없이 실행됩니다. 선택 실습에서만 Gemini API를 호출합니다.

> 이 노트북의 문서는 수업 실습용 가상 데이터입니다. 실제 강의계획서 내용이 아닙니다.

## 1. 수업용 문서와 검색 함수

In [1]:
# 검색할 문서를 간단한 리스트로 준비합니다. (가상 데이터)
DOCUMENTS = [
    {"id": "D1", "text": "지능시스템 수업의 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%입니다."},
    {"id": "D2", "text": "지능시스템 과제는 Colab 노트북을 제출하며, 제출 기한은 수업 공지로 안내합니다."},
    {"id": "D3", "text": "지능시스템 수업에서는 RAG, Workflow, Agent Harness를 실습합니다."},
    {"id": "D4", "text": "AI 활용은 과제 지침에서 허용 범위를 확인하고 사용 사실을 명시해야 합니다."},
]
QUESTION = "지능시스템 과제의 평가 비율과 제출 방법은?"

# 실제 벡터 DB 대신 규칙 기반 검색을 구현합니다.
# query에 포함된 단어와 문서 내용의 단어가 얼마나 겹치는지 계산합니다.
def retrieve(query):
    keywords = [w for w in query.replace('?', '').split() if len(w) >= 2]
    ranked = []
    for doc in DOCUMENTS:
        score = sum(1 for word in keywords if word in doc['text'])
        if score > 0:
            ranked.append((score, doc))
    ranked.sort(key=lambda pair: -pair[0])
    return [doc for _, doc in ranked[:1]]  # 검색 1회당 가장 관련된 문서 1개

print('질문:', QUESTION)
print('1회 검색 결과:', retrieve(QUESTION))

질문: 지능시스템 과제의 평가 비율과 제출 방법은?
1회 검색 결과: [{'id': 'D1', 'text': '지능시스템 수업의 평가는 과제 20%, 중간고사 30%, 기말고사 30%, 출석 20%입니다.'}]


## 2. 답변 생성 함수 (API 없이 동작)

In [2]:
def generate(question, docs):
    """검색된 문서에서 관련 항목만 간단하게 정리합니다."""
    combined = " ".join(d['text'] for d in docs)
    parts = []
    if '과제 20%' in combined:
        parts.append('과제 평가 비율은 20%입니다.')
    if 'Colab 노트북' in combined:
        parts.append('과제는 Colab 노트북으로 제출합니다.')
    return ' '.join(parts) if parts else '검색된 문서만으로 답할 수 없습니다.'

def fixed_rag(question, retrieve, generate):
    # Fixed RAG의 경로는 언제나 '검색 1회 → 답변 생성'입니다.
    docs = retrieve(question)
    return generate(question, docs)

fixed_answer = fixed_rag(QUESTION, retrieve, generate)
print('[Fixed RAG] 결과:', fixed_answer)

[Fixed RAG] 결과: 과제 평가 비율은 20%입니다.


## 3. Workflow RAG: rewrite, judge, 반복 검색

In [3]:
def rewrite(question):
    # 학생이 흐름을 이해하도록 검색어를 간단히 정규화합니다.
    return '지능시스템 과제'

def judge(question, collected):
    """문서가 충분한지 평가하고, 부족하면 다음 검색어를 지정합니다."""
    all_text = ' '.join(d['text'] for d in collected)
    has_grade = '과제 20%' in all_text
    has_submit = 'Colab 노트북' in all_text
    enough = has_grade and has_submit

    if not has_grade:
        next_query = '평가 비율'
    elif not has_submit:
        next_query = '제출 Colab'
    else:
        next_query = ''
    return enough, next_query

def workflow_rag(question, rewrite, retrieve, judge, generate, max_rounds=3):
    query = rewrite(question)
    collected = []
    log = []

    # 반복 횟수와 실행 순서는 개발자가 고정합니다.
    for round_id in range(1, max_rounds + 1):
        docs = retrieve(query)
        collected.extend(docs)
        enough, next_query = judge(question, collected)
        log.append({
            'round': round_id, 'query': query,
            'doc_ids': [doc['id'] for doc in docs],
            'enough': enough, 'next_query': next_query,
        })
        if enough:
            break
        query = next_query

    # 같은 문서가 여러 번 수집되었으면 중복 제거합니다.
    unique_docs = {doc['id']: doc for doc in collected}
    answer = generate(question, list(unique_docs.values()))
    return answer, log, enough

workflow_answer, workflow_log, enough = workflow_rag(
    QUESTION, rewrite, retrieve, judge, generate, max_rounds=3
)
print('[Workflow RAG] 결과:', workflow_answer)
print('정보 충분?', enough)
for row in workflow_log:
    print(row)

[Workflow RAG] 결과: 과제 평가 비율은 20%입니다. 과제는 Colab 노트북으로 제출합니다.
정보 충분? True
{'round': 1, 'query': '지능시스템 과제', 'doc_ids': ['D1'], 'enough': False, 'next_query': '제출 Colab'}
{'round': 2, 'query': '제출 Colab', 'doc_ids': ['D2'], 'enough': True, 'next_query': ''}


## 4. 두 결과를 비교하고 검증하기

Fixed RAG는 검색을 **정확히 한 번** 수행합니다. Workflow RAG는 `judge()` 결과에 따라 같은 **고정된 루프**를 최대 3번 실행합니다.

In [4]:
# 실습이 제대로 구현되었는지 자동으로 확인합니다.
assert '과제 평가 비율' in fixed_answer or 'Colab 노트북' in fixed_answer
assert '과제 평가 비율은 20%' in workflow_answer
assert 'Colab 노트북' in workflow_answer
assert enough is True
assert 1 <= len(workflow_log) <= 3
assert len(workflow_log) > 1, '이 예시는 반복 검색이 실제로 발생해야 합니다.'
print('✅ Fixed RAG와 Workflow RAG 비교 테스트 통과')

# 반복이 1회로 제한되면 필요한 근거를 모두 찾지 못할 수 있습니다.
short_answer, short_log, short_enough = workflow_rag(
    QUESTION, rewrite, retrieve, judge, generate, max_rounds=1
)
print('1회 제한:', short_answer)
print('정보 충분?', short_enough)
assert len(short_log) == 1

✅ Fixed RAG와 Workflow RAG 비교 테스트 통과
1회 제한: 과제 평가 비율은 20%입니다.
정보 충분? False


## 5. `judge()`를 LLM으로 바꾸면 Agent인가?

**아닙니다.** `judge`가 모델에 의해 동적으로 결정을 내려도, 개발자가 `rewrite → retrieve → judge → 최대 3회 반복 → generate`라는 **실행 구조(topology)**를 사전에 정했기 때문입니다.

아래 선택 실습은 **평가 단계만 Gemini LLM으로 교체**합니다. API 호출을 원하지 않으면 건너뛰어도 됩니다. 모델 답변은 코드에서 검증하고, 실패하면 규칙 기반 `judge()`를 사용합니다.

### 5-1. (선택) Colab Secret 설정

1. Colab 왼쪽의 **비밀정보(열쇠)**에 `GEMINI_API_KEY`를 등록하고 노트북 접근을 허용합니다.
2. 다음 셀의 `USE_GEMINI = True`로 바꾸면 실제 API를 호출합니다.
3. 모델 ID는 계정에서 지원되는 모델로 설정합니다. 기본값은 `gemini-3.5-flash-lite`입니다.

**주의:** API 요청에는 요금 또는 무료 사용량 제한이 적용될 수 있습니다.

In [5]:
# 선택 실습: 기본값 False이므로 API 키 없이도 전체 노트북이 실행됩니다.
USE_GEMINI = False
GEMINI_MODEL = 'gemini-3.5-flash-lite'

if USE_GEMINI:
    %pip -q install -U google-genai

In [7]:
import json

def gemini_judge(question, collected):
    """Gemini에게 충분성 평가를 요청하되, 반환값은 Python 코드가 검증합니다."""
    # 네트워크 호출은 선택 실습에서만 수행됩니다.
    from google import genai
    from google.genai import types
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
    except (ImportError, KeyError):
        api_key = None
    if not api_key:
        raise ValueError('Colab Secret GEMINI_API_KEY를 등록하세요.')

    client = genai.Client(api_key=api_key)
    docs_text = '\n'.join(d['text'] for d in collected)
    prompt = f"""질문: {question}
검색된 문서:
{docs_text}
질문에 답하기에 근거가 충분한지 판정하세요.
반드시 JSON 객체만 출력하세요.
예: {{"enough": false, "next_query": "평가 비율"}}
충분하면 enough=true, next_query=""로 답하세요.
부족하면 enough=false 및 다음 검색어를 제시하세요."""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(response_mime_type='application/json', temperature=0),
    )
    obj = json.loads(response.text)
    if not isinstance(obj, dict) or type(obj.get('enough')) is not bool:
        raise ValueError('LLM의 충분성 판정 형식이 잘못되었습니다.')
    next_query = obj.get('next_query', '')
    if not isinstance(next_query, str):
        raise ValueError('next_query는 문자열이어야 합니다.')
    if not obj['enough'] and not next_query.strip():
        raise ValueError('정보가 부족할 때는 다음 검색어가 필요합니다.')
    return obj['enough'], next_query


def safe_judge(question, collected):
    if not USE_GEMINI:
        return judge(question, collected)
    try:
        return gemini_judge(question, collected)
    except Exception as exc:
        # 장애 시 학습이 중단되지 않도록 예외를 보여주고 기본 평가로 복귀합니다.
        print('LLM 평가 실패 → 규칙 기반 평가 사용:', type(exc).__name__, str(exc)[:160])
        return judge(question, collected)

llm_workflow_answer, llm_workflow_log, llm_enough = workflow_rag(
    QUESTION, rewrite, retrieve, safe_judge, generate, max_rounds=3
)
print('선택 모드 사용:', 'Gemini 평가' if USE_GEMINI else '규칙 기반 평가')
print('답변:', llm_workflow_answer)
print('실행한 반복 수:', len(llm_workflow_log))
for row in llm_workflow_log:
    print(row)

선택 모드 사용: 규칙 기반 평가
답변: 과제 평가 비율은 20%입니다. 과제는 Colab 노트북으로 제출합니다.
실행한 반복 수: 2
{'round': 1, 'query': '지능시스템 과제', 'doc_ids': ['D1'], 'enough': False, 'next_query': '제출 Colab'}
{'round': 2, 'query': '제출 Colab', 'doc_ids': ['D2'], 'enough': True, 'next_query': ''}


## 6. 학생 실습 과제

1. `max_rounds`를 1, 2, 3으로 바꾸고 답변과 검색 로그를 비교해 보세요.
2. 문서 `D2`를 삭제한 뒤 `judge()`가 어떤 검색어를 생성하는지 확인해 보세요.
3. `rewrite()`를 변경해 검색 순서가 바뀌는지 실험해 보세요.
4. (선택) `USE_GEMINI=True`로 바꿔 LLM 평가가 반환한 `enough`와 `next_query`를 비교해 보세요.
5. **토의:** `judge()`가 LLM이어도 왜 Agent가 아니라 Workflow인가요?

### 핵심 정리
- **Fixed RAG:** `retrieve → generate`의 고정 1회 검색
- **Workflow RAG:** `rewrite → retrieve → judge → 최대 N회 반복 → generate`
- **Agent와의 구별:** LLM 사용 유무가 아니라 **실행 경로를 누가 결정하는지**가 중요합니다.

> 교육용 단순 검색 함수는 키워드 겹침으로 동작합니다. 실제 검색 성능을 대표하지는 않습니다. 또한 누적 문서가 충분하지 않아도 반복 한도에 도달하면 답변 함수가 호출되므로, 서비스에서는 별도의 근거 부족 표시·안전한 응답 처리도 필요합니다.